# Notebook 4: Model Training

**Purpose:** This notebook trains the AtrionNet v4.0 Hybrid model on the LUDB dataset.
It implements the full research training loop including:
- Multi-task loss (Focal + SmoothL1 + Dice)
- AdamW optimizer with cosine annealing warm restarts
- Per-epoch validation with F1 and mAP calculation
- Best-model checkpointing and periodic epoch checkpoints
- Automatic generation of a final research report

**Expected time:** ~30-60 minutes on a T4 GPU in Google Colab.

## Step 1: Setup — Paths, Imports, and Device

In [ ]:
import sys
import os
import random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm

PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Step 2: Training Configuration

All hyperparameters are defined in one place for transparency and easy modification.

In [ ]:
CONFIG = {
    'data_dir'               : os.path.join(PROJECT_ROOT, 'data', 'raw', 'ludb'),
    'device'                 : DEVICE,
    'epochs'                 : 200,    # BiLSTM+Attention needs more epochs to converge
    'batch_size'             : 16,
    'lr'                     : 5e-5,   # Low LR for stable temporal learning
    'weight_decay'           : 1e-4,
    'early_stopping_patience': 30,     # High patience to survive precision plateaus
    'seed'                   : 42,
    'plot_dir'               : os.path.join(PROJECT_ROOT, 'reports', 'plots'),
    'model_save_path'        : os.path.join(PROJECT_ROOT, 'weights', 'atrion_hybrid_best.pth'),
    'checkpoint_dir'         : os.path.join(PROJECT_ROOT, 'weights'),
    'test_indices_path'      : os.path.join(PROJECT_ROOT, 'data', 'processed', 'test_indices.npy'),
}

os.makedirs(CONFIG['plot_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, 'data', 'processed'), exist_ok=True)

print("Configuration loaded.")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Step 3: Fix Random Seed for Reproducibility

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"Seed set to {CONFIG['seed']}.")

## Step 4: Load Data

Load all LUDB records and split into Train / Validation / Test sets.

In [ ]:
from src.data_pipeline.ludb_loader import LUDBLoader
from src.data_pipeline.instance_dataset import AtrionInstanceDataset

loader = LUDBLoader(CONFIG['data_dir'])
signals, annotations = loader.get_all_data()

indices = np.arange(len(signals))
np.random.shuffle(indices)

tr_stop  = int(0.70 * len(indices))
val_stop = int(0.85 * len(indices))

idx_tr   = indices[:tr_stop]
idx_val  = indices[tr_stop:val_stop]
idx_test = indices[val_stop:]

np.save(CONFIG['test_indices_path'], idx_test)

train_ds = AtrionInstanceDataset(signals[idx_tr],  [annotations[i] for i in idx_tr],  is_train=True)
val_ds   = AtrionInstanceDataset(signals[idx_val], [annotations[i] for i in idx_val], is_train=False)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'])

print(f"Train: {len(idx_tr)} | Val: {len(idx_val)} | Test: {len(idx_test)} records")

## Step 5: Build the Model, Optimizer, and Scheduler

- **AdamW**: A robust optimizer with weight decay to prevent overfitting on a small dataset.
- **CosineAnnealingWarmRestarts**: The learning rate decays smoothly then restarts, which helps escape local minima.

In [ ]:
from src.modeling.atrion_net import AtrionNetHybrid

model     = AtrionNetHybrid(in_channels=12).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model ready. Total parameters: {total_params:,}")

## Step 6: Run the Training Loop

The loop runs for up to `CONFIG['epochs']` steps. Each epoch:
1. Trains on all training batches and accumulates loss.
2. Runs validation — computes F1, mAP, Precision, and Recall.
3. Saves the best model when F1 improves.
4. Saves a checkpoint every 20 epochs.
5. Stops early if no improvement for 30 consecutive epochs.

In [ ]:
from src.losses.segmentation_losses import create_instance_loss
from src.engine.atrion_evaluator import compute_instance_metrics, calculate_mAP

history = {
    'train_loss': [], 'val_loss': [], 'lr': [],
    'val_precision': [], 'val_recall': [], 'val_f1': [], 'val_map': []
}

best_f1         = 0.0
patience_counter = 0
best_pr         = None

print(f"Starting training on {DEVICE} | Epochs: {CONFIG['epochs']} | Early stopping patience: {CONFIG['early_stopping_patience']}")

for epoch in range(CONFIG['epochs']):

    # --- Train ---
    model.train()
    t_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}", leave=False):
        sigs  = batch['signal'].to(DEVICE)
        targs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'signal'}

        optimizer.zero_grad()
        preds = model(sigs)
        loss  = create_instance_loss(preds, targs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping for stability
        optimizer.step()
        t_loss += loss.item()

    # --- Validate ---
    model.eval()
    v_loss = 0.0
    all_tp_lists, all_scores, total_gt = [], [], 0
    v_prec, v_rec, v_f1 = [], [], []

    with torch.no_grad():
        val_idx_counter = 0
        for batch in val_loader:
            sigs  = batch['signal'].to(DEVICE)
            targs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'signal'}
            out   = model(sigs)
            v_loss += create_instance_loss(out, targs).item()

            for i in range(len(sigs)):
                actual_idx = idx_val[val_idx_counter]
                targets = [{'span': (o, f)} for o, p, f in annotations[actual_idx]['p_waves']]
                res = compute_instance_metrics(
                    out['heatmap'][i:i+1].cpu().numpy(),
                    out['width'][i:i+1].cpu().numpy(),
                    targets
                )
                all_tp_lists.append(res['tp_list'])
                all_scores.append(res['scores'])
                total_gt += res['n_gt']
                v_prec.append(res['precision'])
                v_rec.append(res['recall'])
                v_f1.append(res['f1'])
                val_idx_counter += 1

    m_ap, val_rec_curve, val_pre_curve = calculate_mAP(all_tp_lists, all_scores, total_gt)
    avg_f1 = float(np.mean(v_f1))

    # Record history
    history['train_loss'].append(t_loss / len(train_loader))
    history['val_loss'].append(v_loss / len(val_loader))
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['val_precision'].append(float(np.mean(v_prec)))
    history['val_recall'].append(float(np.mean(v_rec)))
    history['val_f1'].append(avg_f1)
    history['val_map'].append(float(m_ap))

    print(f"[Epoch {epoch+1:03d}] TrainLoss: {history['train_loss'][-1]:.4f} | ValLoss: {history['val_loss'][-1]:.4f} | F1: {avg_f1:.4f} | mAP: {m_ap:.4f}")

    # Save best model
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        torch.save(model.state_dict(), CONFIG['model_save_path'])
        best_pr = (val_rec_curve, val_pre_curve, m_ap)
        patience_counter = 0
        print(f"  --> Best model saved (F1={best_f1:.4f})")
    else:
        patience_counter += 1

    # Save periodic checkpoint every 20 epochs
    if (epoch + 1) % 20 == 0:
        cp_path = os.path.join(CONFIG['checkpoint_dir'], f'checkpoint_epoch_{epoch+1}.pth')
        torch.save({
            'epoch'               : epoch + 1,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1'              : avg_f1,
        }, cp_path)
        print(f"  --> Checkpoint saved: {cp_path}")

    scheduler.step()

    # Early stopping check
    if patience_counter >= CONFIG['early_stopping_patience']:
        print(f"Early stopping triggered at epoch {epoch+1}.")
        break

print(f"\nTraining complete. Best Validation F1: {best_f1:.4f}")

## Step 7: Save the Training History

We save the history dictionary so the visualization notebook can load it without re-training.

In [ ]:
import json

history_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'training_history.json')

# Convert numpy floats to plain Python floats for JSON serialization
serializable_history = {k: [float(x) for x in v] for k, v in history.items()}
with open(history_path, 'w') as f:
    json.dump(serializable_history, f, indent=2)

print(f"Training history saved to: {history_path}")

## Step 8: Generate the Final Research Report

A plain text report summarising all key results from this training run.

In [ ]:
report_path = os.path.join(CONFIG['plot_dir'], 'research_report.txt')

with open(report_path, 'w') as f:
    f.write("====================================================\n")
    f.write("ATRION-NET v4.0 FINAL RESEARCH REPORT\n")
    f.write("====================================================\n\n")
    f.write("MODEL ARCHITECTURE: Attentional-Hybrid (Inception + BiLSTM + SE-Attention)\n")
    f.write(f"DEVICE: {CONFIG['device']}\n")
    f.write(f"TOTAL EPOCHS TRAINED: {len(history['train_loss'])}\n")
    f.write(f"BEST VALIDATION F1: {best_f1:.4f}\n")
    best_epoch = int(np.argmax(history['val_f1']))
    f.write(f"BEST mAP @ 0.5: {history['val_map'][best_epoch]:.4f}\n")
    f.write(f"BEST PRECISION: {history['val_precision'][best_epoch]:.4f}\n")
    f.write(f"BEST RECALL: {history['val_recall'][best_epoch]:.4f}\n\n")
    f.write("--- DATASET SUMMARY ---\n")
    f.write(f"Training Samples  : {len(idx_tr)}\n")
    f.write(f"Validation Samples: {len(idx_val)}\n")
    f.write(f"Test Samples      : {len(idx_test)}\n\n")
    f.write("--- HYPERPARAMETERS ---\n")
    for k, v in CONFIG.items():
        f.write(f"  {k}: {v}\n")

print(f"Research report saved to: {report_path}")

# Also print it here
with open(report_path) as f:
    print(f.read())

---
**Training complete. Proceed to `05_evaluation/model_evaluation.ipynb`.**